# 데이터 EDA & 전처리

---

In [29]:
# 데이터 호출

data <- read.csv(
  "data.csv",
  stringsAsFactors = FALSE,
  na.strings = c("", "NA", "N/A")
)

In [30]:
dim(data)
head(data)

[1] 71351    32

,spkid,full_name,name,neo,pha,sats,H,G,PC,diameter,⋯,n,tp,tp_cal,per,per_y,moid,moid_ld,class,moid_jup,equinox
,<int>,<chr>,<chr>,<chr>,<chr>,<int>,<dbl>,<dbl>,<lgl>,<dbl>,⋯,<dbl>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<dbl>,<chr>
1,20000132,132 Aethra (A873 LA),Aethra,N,N,0,8.96,NA,NA,42.870,⋯,0.2335,2460516,2024-07-24.7,1540,4.22,0.782,304.0,MCA,2.21,J2000
2,20000391,391 Ingeborg (A894 VB),Ingeborg,N,N,0,10.85,NA,NA,15.751,⋯,0.2788,2460826,2025-05-30.0,1290,3.54,0.645,251.0,MCA,2.54,J2000
3,20000433,433 Eros (A898 PA),Eros,Y,N,0,10.39,0.46,NA,16.840,⋯,0.5598,2461089,2026-02-17.3,643,1.76,0.148,57.7,AMO,3.29,J2000
4,20000475,475 Ocllo (A901 PA),Ocllo,N,N,0,11.54,NA,NA,17.830,⋯,0.2360,2461442,2027-02-05.8,1530,4.18,0.674,262.0,MCA,2.10,J2000
5,20000512,512 Taurinensis (A903 MC),Taurinensis,N,N,0,10.76,NA,NA,23.090,⋯,0.3043,2461448,2027-02-11.6,1180,3.24,0.651,253.0,MCA,2.72,J2000
6,20000699,699 Hela (A910 LC),Hela,N,N,0,11.45,NA,NA,NA,⋯,0.2331,2460668,2024-12-23.3,1540,4.23,0.628,244.0,MCA,2.10,J2000


In [31]:
data <- read.csv(
  "data.csv",
  stringsAsFactors = FALSE,
  na.strings = c("", "NA", "N/A")
)

# feature별 기본 요약
feature_summary <- data.frame(
  feature = names(data),
  type = sapply(data, function(x) class(x)[1]),
  n = nrow(data),
  missing_n = sapply(data, function(x) sum(is.na(x))),
  missing_pct = round(sapply(data, function(x) mean(is.na(x)) * 100), 2),
  unique_n = sapply(data, function(x) length(unique(x[!is.na(x)]))),
  stringsAsFactors = FALSE
)

feature_summary$constant <- feature_summary$unique_n <= 1

feature_summary

,feature,type,n,missing_n,missing_pct,unique_n,constant
,<chr>,<chr>,<int>,<int>,<dbl>,<int>,<lgl>
spkid,spkid,integer,71351,0,0.00,71351,FALSE
full_name,full_name,character,71351,0,0.00,71351,FALSE
name,name,character,71351,70882,99.34,469,FALSE
neo,neo,character,71351,0,0.00,2,FALSE
pha,pha,character,71351,773,1.08,2,FALSE
sats,sats,integer,71351,0,0.00,3,FALSE
H,H,numeric,71351,58,0.08,1859,FALSE
G,G,numeric,71351,71337,99.98,12,FALSE
PC,PC,logical,71351,71351,100.00,0,TRUE


In [32]:
# 제거할 feature 지정
drop_cols <- c(
  "spkid", "full_name", "name",       # 식별자: 천체 이름/번호라 예측 feature로 부적절

  "G", "PC", "diameter", "extent",
  "albedo", "rot_per", "BV",          # 결측률 90% 이상: na.omit 전에 열 단위 제거

  "orbit_id",                         # 내부 궤도해 식별자: 물리/궤도 측정값이 아님

  "equinox",                          # 모든 값이 동일한 상수 feature

  "tp",                               # 근일점 통과 시각: 현재 목적의 핵심 궤도 형상 feature로 보기 어려움

  "tp_cal",                           # tp의 달력형 변환값

  "moid",                             # 궤도 요소로부터 계산되는 값이며 PHA 정의에 직접 사용됨

  "moid_ld",                          # moid의 단위 변환값

  "class",                            # 궤도 요소를 바탕으로 붙은 분류 라벨

  "q", "ad",                          # a, e로부터 유도 가능: q = a * (1 - e), ad = a * (1 + e)

  "n", "per", "per_y",                # 공전 속도/공전 주기 계열 중복: n = 360 / per, per_y는 per 단위 변환

  "H"                                 # PHA 정의에 직접 들어가므로 leakage 방지 목적에서 제거
)

# 실제 데이터에 존재하는 열만 제거 대상으로 사용
drop_cols <- drop_cols[drop_cols %in% names(data)]

# 제거된 feature 출력
cat("제거할 feature:\n")
print(drop_cols)

# feature 제거
data_clean <- data[, !(names(data) %in% drop_cols)]

# 남은 feature 출력
cat("\n남은 feature:\n")
print(names(data_clean))

# 남은 데이터에서 결측치가 있는 행 제거
data_clean <- na.omit(data_clean)

# 최종 데이터 크기 확인
dim(data_clean)

제거할 feature:
 [1] "spkid"     "full_name" "name"      "G"         "PC"        "diameter" 
 [7] "extent"    "albedo"    "rot_per"   "BV"        "orbit_id"  "equinox"  
[13] "tp"        "tp_cal"    "moid"      "moid_ld"   "class"     "q"        
[19] "ad"        "n"         "per"       "per_y"     "H"        

남은 feature:
[1] "neo"      "pha"      "sats"     "e"        "a"        "i"        "om"      
[8] "w"        "moid_jup"


[1] 70578     9

In [33]:
write.csv(
  data_clean,
  "data_clean.csv",
  row.names = FALSE
)

---